# Running the model --- FEWS farm model

Jones (2022), *Environment Systems and Decisions*,
[10.1007/s10669-021-09838-8](https://doi.org/10.1007/s10669-021-09838-8).

**This notebook runs the model.** It needs `gurobipy`, and should need no licence
file: the models it solves are 37 variables against a documented 2,000-variable
cap on the licence bundled with `pip install gurobipy`. That cap is reported
rather than measured here, so treat it as an inference until this has been run
somewhere without a licence.

**It is not the reproduction claim.** `00_verification.ipynb` makes that one and
needs nothing but numpy. This one shows that the code runs and behaves.

**Nothing here is a reduced instance.** The collapsed model solves the *whole*
problem: given the capacities nothing couples one run-year to another, and
precipitation takes five values, so 704,002 variables become 37 carrying integer
weights. Same mathematics --- `tests/test_collapsed_agrees.py` asserts the two
agree. So there is no reduction to stamp onto a figure, and a warning here would
be a false one.

**This notebook is thin.** It imports the package and calls it; the model lives
in `src/fews_stochopt/`.

## 1. Install

On Colab, **clone the repository and install from that checkout.**
`pip install git+https://...` is not enough and fails in a way that looks like a
bug in the model: it installs the package but not `data/`, because the
precipitation inputs live in `data/raw/` and are not part of the wheel. The
installed `config.py` then derives the repository root relative to site-packages,
and the first cell that loads data dies with a `FileNotFoundError` naming a
directory that has nothing to do with the problem.

In [1]:
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = "https://github.com/sear-labs/fews-stochopt-esd-2022.git"
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    "fews-stochopt-esd-2022"], check=True)
    print("cloned and installed from the checkout")
else:
    print("running from a local checkout; `pip install -e .` once if you have not")

import fews_stochopt

print("fews_stochopt", fews_stochopt.__version__)

running from a local checkout; `pip install -e .` once if you have not
fews_stochopt 1.0.0


## 2. How big is the model, really

In [2]:
import pathlib

import fews_stochopt

# Derived from the INSTALLED package, never from the working directory. A
# relative "../scripts/..." breaks the moment the notebook runs from anywhere but
# its own folder, which is exactly what happens on Colab after the clone.
ROOT = pathlib.Path(fews_stochopt.__file__).resolve().parents[2]


def run_script(name, *args):
    """Run a repository script and show BOTH streams and the return code.

    Printing only stdout is how a failing script comes to look like one that did
    nothing.
    """
    proc = subprocess.run(
        [sys.executable, str(ROOT / "scripts" / name), *args],
        capture_output=True, text=True, cwd=str(ROOT),
    )
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.stderr:
        print("--- stderr ---")
        print(proc.stderr.rstrip())
    if proc.returncode != 0:
        raise RuntimeError(f"{name} exited {proc.returncode}")
    # Return the code, not the CompletedProcess. Its repr carries `args`, which
    # is the interpreter path and the absolute script path -- so a cell ending
    # in this call committed one machine's home directory into the notebook.
    return proc.returncode


import gurobipy as gp

from fews_stochopt import collapsed, load_config
from fews_stochopt.model import EXPECTED_VALUE, KNOWN_CLIMATE, STOCHASTIC

cfg = load_config()
env = gp.Env(params={"OutputFlag": 0})
weights, n_runs = collapsed.rain_weights(cfg, "EP")
probe = collapsed.build(cfg, weights, n_runs, env)
print(f"collapsed model: {probe.NumVars} variables, {probe.NumConstrs} linear "
      f"and {probe.NumQConstrs} quadratic constraints")
inside = "well inside" if probe.NumVars <= 2000 else "OVER"
print(f"documented cap on the bundled licence: 2,000 variables -> {inside} "
      f"({probe.NumVars} declared; not verified under that licence here)")
del probe, env

collapsed model: 37 variables, 25 linear and 5 quadratic constraints
documented cap on the bundled licence: 2,000 variables -> well inside (37 declared; not verified under that licence here)


## 3. Solve it, and compare against the full pipeline

Three of the four scenarios collapse and solve in milliseconds. **Perfect
Information does not collapse** --- every run picks its own capacities, so the
runs share nothing to aggregate over. It is 4,000 separate solves of 178
variables, about two minutes, and it is skipped here.

In [3]:
import pandas as pd

QUICK = True   # skip the 4,000-solve Perfect Information scenario

rows = []
for site in cfg.sites:
    for scenario in (STOCHASTIC, KNOWN_CLIMATE, EXPECTED_VALUE):
        result = collapsed.solve(cfg, site, scenario)
        rows.append({"site": site, "label": cfg.site(site).label,
                     "scenario": scenario, "mean_profit": result["objective"],
                     "alt_water_cap_cm": result["alt_water_cap"],
                     "alt_elc_cap_kW": result["alt_elc_cap"]})
solved = pd.DataFrame(rows)

stats = pd.read_csv(ROOT / "results" / "clean" / "scenario_stats.csv")
full = stats[stats["variable"] == "profit_mean"].set_index(["label", "scenario"])["value"]
solved["full_pipeline"] = [full[(r.label, r.scenario)] for r in solved.itertuples()]
solved["difference"] = solved["mean_profit"] - solved["full_pipeline"]

# Fixed uuid and HTML(), as in the verification notebook: the two tokens a
# bare Styler emits are the only thing that was not reproducible here.
from IPython.display import HTML

styled = (solved[["label", "scenario", "full_pipeline", "mean_profit", "difference"]]
          .style.format({"full_pipeline": "{:,.4f}", "mean_profit": "{:,.4f}",
                         "difference": "{:+.4f}"}).set_uuid("solved"))
display(HTML(styled.to_html()))

if QUICK:
    print("QUICK = True: Perfect Information was not solved here.")

,label,scenario,full_pipeline,mean_profit,difference
0,Equally Probable,Stochastic,"2,246,937.8910","2,246,937.9654",+0.0744
1,Equally Probable,"Known Climate, Unknown Weather","2,345,267.1068","2,345,267.1351",+0.0283
2,Equally Probable,Expected Value,"2,246,937.5497","2,246,937.7193",+0.1696
3,Dry Most Likely,Stochastic,"1,899,221.1646","1,899,221.2526",+0.0880
4,Dry Most Likely,"Known Climate, Unknown Weather","1,964,087.4288","1,964,087.5963",+0.1675
5,Dry Most Likely,Expected Value,"1,898,256.2783","1,898,261.6792",+5.4009


QUICK = True: Perfect Information was not solved here.


## 4. Change something, and see what it does

This is what an example notebook is for. The capacity cost is a knob; halving it
should buy more alternative water.

In [4]:
import copy

base = collapsed.solve(cfg, "DML", STOCHASTIC)

cheaper = copy.deepcopy(cfg)
cheaper.cost_alt_water = cfg.cost_alt_water / 2
changed = collapsed.solve(cheaper, "DML", STOCHASTIC)

header = f"{'':22}{'water cap (cm)':>16}{'elc cap (kW)':>15}{'mean profit':>16}"
print(header)
print(f"{'as published':22}{base['alt_water_cap']:>16.4f}"
      f"{base['alt_elc_cap']:>15.4f}{base['objective']:>16,.2f}")
print(f"{'half the water cost':22}{changed['alt_water_cap']:>16.4f}"
      f"{changed['alt_elc_cap']:>15.4f}{changed['objective']:>16,.2f}")

assert changed["alt_water_cap"] > base["alt_water_cap"], (
    "halving the cost of alternative water should buy more of it")
print("More alternative water, as it should be.")

                        water cap (cm)   elc cap (kW)     mean profit
as published                   10.9549       206.7938    1,899,221.25
half the water cost            17.0100       309.6474    2,101,643.59
More alternative water, as it should be.


## 5. The full 704,002-variable formulation --- needs a real licence

`scripts/run_all.py` solves the model as the published run wrote it, reproduces
both of the paper's tables, and takes about eight minutes.

On Colab a node-locked licence cannot work --- the virtual machine differs every
session --- so use WLS credentials held as Colab secrets, never as literals: a
key committed to a repository is exposed the moment the repository is shared, and
deleting it later does not remove it from the history.

```python
import os
from google.colab import userdata          # SecretNotFoundError is EXPECTED
for k in ("WLSACCESSID", "WLSSECRET", "LICENSEID"):
    os.environ["GRB_" + k] = userdata.get("GRB_" + k)
```

`tests/test_collapsed_agrees.py` ties the two together: it solves the collapsed
model and the full one and asserts they agree, so the fast path above is not a
different model with a convenient answer.